In [ ]:
import requests
from xml.etree import ElementTree as ET
import time
from tqdm import tqdm
import os
from dotenv import load_dotenv
load_dotenv()

BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

pmids = []
for year in range(2020, 2026):
    params = {
        "db": "pubmed",
        "term": '("Genome-Wide Association Study"[MeSH]) AND ("Alzheimer Disease"[MeSH]) AND (free full text[sb])',
        "datetype": "pdat",
        "mindate": f"{year}/01/01",
        "maxdate": f"{year}/12/31",
        "retmode": "json",
        "retmax": 10000,
        "api_key": os.environ.get("ENTREZ_API_KEY", "")
    }

    r = requests.get(f"{BASE}/esearch.fcgi", params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    curr_year_pmids = data["esearchresult"]["idlist"]
    if curr_year_pmids:
        pmids.extend(curr_year_pmids)
    time.sleep(1)
print(len(pmids))
# print("PMIDs:", pmids)

pmids_with_pmcids = []
if pmids:
    with open("gwas_pmid_pmcid.txt", "w") as f:
        for i in tqdm(range(0, len(pmids), 10)):
            try:
                batch_pmids = pmids[i: min(i + 10, len(pmids))]
                # rate limit is 10 per sec => batching + sleep after request
                url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"
                params = {
                    "dbfrom": "pubmed",
                    "db": "pmc",
                    "id": batch_pmids,
                    "retmode": "xml"
                }
                
                r = requests.get(url, params=params, timeout=30)
                r.raise_for_status()
                root = ET.fromstring(r.content)
                
                for linkset in root.findall("./LinkSet"):
                    pmid_elems = linkset.findall("./IdList/Id")
                    if not pmid_elems:
                        continue
                    pmid = pmid_elems[0].text
                    pmcids = []
                    for db in linkset.findall("./LinkSetDb"):
                        dbto = db.find("DbTo")
                        if dbto is not None and dbto.text == "pmc":
                            pmcids.extend([x.text for x in db.findall("./Link/Id")])
                    if len(pmcids) > 0:
                        f.write(f"{pmid},{pmcids[0]}\n")
                        pmids_with_pmcids.append((pmid, pmcids[0]))
            except Exception as e:
                continue
print(len(pmids_with_pmcids))

In [ ]:
def get_mesh_terms(pmid):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed", 
        "id": pmid, 
        "rettype": "xml", 
        "retmode": "xml", 
        "api_key": os.environ.get("ENTREZ_API_KEY", "")
    }
    
    r = requests.get(url, params=params)
    root = ET.fromstring(r.content)
    
    mesh_terms = []
    for heading in root.findall(".//MeshHeading"):
        descriptor = heading.find("DescriptorName").text
        qualifiers = [q.text for q in heading.findall("QualifierName")]
        mesh_terms.append({"descriptor": descriptor, "qualifiers": qualifiers})
    
    return mesh_terms

for filename in os.listdir("pred_tables"):
    if ".csv" in filename:
        pmid = int(filename.split("_")[0])
        print(pmid)
        curr_mesh_terms = get_mesh_terms(pmid)
        print(curr_mesh_terms)
        print([term["descriptor"] for term in curr_mesh_terms])